In [ ]:
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.isdir("smartstock-ai"):          # Colab/Kaggle: clone your repo first
    subprocess.run(["git", "clone", "https://github.com/<you>/smartstock-ai.git"], check=True); os.chdir("smartstock-ai")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "training/requirements.txt"], check=True)
ROOT = os.getcwd()
while not os.path.isdir(os.path.join(ROOT, "backend")) and ROOT != "/": ROOT = os.path.dirname(ROOT)
os.chdir(ROOT); sys.path[:0] = [os.path.join(ROOT, "backend"), os.path.join(ROOT, "training", "scripts")]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
RAW_OK = os.path.exists("data/raw/m5/sales_train_validation.csv")
if not RAW_OK:   # no real M5 download present -> generate M5-format synthetic data so the notebook runs anywhere
    subprocess.run([sys.executable, "training/scripts/make_synthetic_m5.py"], check=True)
if not os.path.exists("data/processed/panel.npz"):
    subprocess.run([sys.executable, "training/scripts/prepare_data.py"], check=True)
def ensure_trained():
    if not os.path.exists("training/outputs/metrics.json"):
        subprocess.run([sys.executable, "training/scripts/train.py"], check=True)

# 06 · Uncertainty — P10 / P50 / P90 quantile models and coverage

In [ ]:
import json; ensure_trained(); m = json.load(open("training/outputs/metrics.json")); print({k: round(v, 3) for k, v in m["uncertainty"].items()})
t = pd.read_parquet("training/outputs/test_predictions.parquet"); s = t[(t.series == t.series.unique()[3]) & (t.origin == t.origin.max())]
plt.fill_between(s.h, s.p10, s.p90, alpha=.25, label="P10–P90"); plt.plot(s.h, s.p50, label="P50"); plt.plot(s.h, s.y, "k.", label="actual"); plt.legend(); plt.title("One series, last test origin"); plt.show()

A P90 forecast should cover ~90% of actuals. Quantiles of *count* data with many zeros are lumpy (the P10 is usually 0), so judge calibration mainly on P50/P90 and the P10–P90 band.